In [3]:
Output = '/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1016. THC Adducts/P1016.3. Analyses/P1016.3.4. Adductome Integration'
cur_date = "111026"

library(readxl)
library(tidyverse)
library(reshape2)
library(org.Hs.eg.db)

# reading in files
results_df = data.frame(read_excel("Input/DESeq2_Results_110726.xlsx"))
thc_adducts_df = data.frame(read_excel("Input/THC_Adducts_111026.xlsx"))
upstream_reg_df = data.frame(read_excel("Input/THCQ_Control_Results_Upstream Regulators_2026-11-10.xlsx"))

In [4]:
head(results_df)
head(thc_adducts_df)
head(upstream_reg_df)

,Gene,log2FC,P.Value,P.Adj,Comparison
,<chr>,<dbl>,<dbl>,<dbl>,<chr>
1,ENSG00000000003,-0.007988413,0.9461493,1,D8THC vs. Control
2,ENSG00000000419,0.075514481,0.6994330,1,D8THC vs. Control
3,ENSG00000000457,0.119709242,0.6461443,1,D8THC vs. Control
4,ENSG00000000460,0.125056396,0.7907686,1,D8THC vs. Control
5,ENSG00000000938,1.111036342,0.4973247,1,D8THC vs. Control
6,ENSG00000000971,0.173825486,0.8245174,1,D8THC vs. Control


,Gene_Name
,<chr>
1,ALB
2,ARAP1
3,ATL3
4,ATP6V0A4
5,CAAP1
6,CARD11


,Regulator,Molecule.Type,P.value,BH.P.value,Activation.z.score,Percentage.overlap,Overlapping.molecules,Total.known.targets
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>
1,YAP1,transcription regulator,1.12e-42,1.32e-38,-1.69,36.86%,286,776
2,RICTOR,other,1.47e-40,8.71e-37,-9.43,50.17%,149,297
3,torin1,chemical reagent,7.83e-39,3.10e-35,-5.79,57.58%,114,198
4,SPEN,transcription regulator,5.95e-31,1.76e-27,6.53,82.81%,53,64
5,beta-estradiol,chemical - endogenous mammalian,3.20e-28,7.59e-25,3.27,23.91%,752,"3,145"
6,HNF4A,transcription regulator,1.69e-27,3.34e-24,1.15,25.13%,590,"2,348"


# Adductome IntegrationXXX
QIAGEN IPA Interpret predicts which upstream regulators are activated or inhibited to explain the up-regulated and down-regulated molecules observed in the dataset. Knowledge of these regulatory mechanisms can help you understand the biological activities occurring in the tissues or cells that are being studied.

 

(For an overview of IPA, see this article; for an overview of IPA Interpret see this article).

 
The term “upstream regulator” refers to any molecule that can affect the expression, transcription, or phosphorylation of another molecule. Upstream regulators encompass the gamut of molecule types found in the literature, from transcription factors to cytokines, microRNAs, receptors, kinases, chemicals, and drugs. Note that phosphorylation information is only used when analyzing phosphoproteomics data.

In [8]:
# getting sig genes
sig_genes_df = results_df %>%
    # filtering for only sig genes
    filter(P.Adj < 0.05 & log2FC > log2(1.5) | P.Adj < 0.05 & log2FC < -log2(1.5)) %>%
    dplyr::rename(ENSEMBL_ID = Gene)

# adding in gene names
sig_genes_df$Gene_Name <- mapIds(org.Hs.eg.db, keys = sig_genes_df$ENSEMBL_ID, keytype = "ENSEMBL", column = "SYMBOL") #adding gene names

# seeing if those gene names correpond to those with adducts post thc exposure
final_sig_df = sig_genes_df %>%
    mutate(Adduct = ifelse(Gene_Name %in% thc_adducts_df$Gene_Name, "Yes", "No"),
          # upstream reugulators??
          Regulator = ifelse(Gene_Name %in% upstream_reg_df$Regulator, "Yes", "No")) 
    

final_sig_df = final_sig_df[,c(1,6,5,7,8,2:4)]

head(final_sig_df)

'select()' returned 1:many mapping between keys and columns



,ENSEMBL_ID,Gene_Name,Comparison,Adduct,Regulator,log2FC,P.Value,P.Adj
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,ENSG00000161055,SCGB3A1,D8THCQ vs. Control,No,No,5.187423,6.699531e-22,1.406165e-17
2,ENSG00000149021,SCGB1A1,D8THCQ vs. Control,No,No,4.153279,2.655023e-18,2.786314e-14
3,ENSG00000124237,CIMIP1,D8THCQ vs. Control,No,No,3.514441,3.981380e-16,1.671304e-12
4,ENSG00000160472,TMEM190,D8THCQ vs. Control,No,No,3.658307,3.763892e-16,1.671304e-12
5,ENSG00000164972,SPMIP6,D8THCQ vs. Control,No,No,3.759643,2.399373e-16,1.671304e-12
6,ENSG00000105519,CAPS,D8THCQ vs. Control,No,No,3.127536,2.038872e-14,7.132315e-11


In [15]:
final_sig_df %>%
    filter(Adduct == 'Yes') %>%
    dplyr::select(Gene_Name) %>%
    unique() %>%
    count()

final_sig_df %>%
    filter(Regulator == 'Yes') %>%
    dplyr::select(Gene_Name) %>%
    unique() %>%
    count()

n
<int>
23


n
<int>
441


Based on this information, there were 23 genes that formed DNA adducts and 441 considered to be upstream regulators after 24hrs of exposure to THCQ.

In [18]:
final_sig_df %>%
    filter(Adduct == 'Yes', Regulator == 'Yes') #%>%
    #dplyr::select(Gene_Name) %>%
    #unique() 

ENSEMBL_ID,Gene_Name,Comparison,Adduct,Regulator,log2FC,P.Value,P.Adj
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
ENSG00000107290,SETX,D8THCQ vs. Control,Yes,Yes,-0.9218473,2.653565e-04,0.0031360177
ENSG00000107290,SETX,D8THCQ vs. D8THC,Yes,Yes,-1.0420981,3.729434e-05,0.0005046879
